# YOLO11n Baseline — Noise Ablation Study v3

**Pipeline dựa trên** `yolov11n_simam_augmentation.ipynb` (Nhóm A1):
- Cùng grouped-stratified split (SEED=42)
- Cùng augmentation mạnh (flipud=0.3, scale=0.5, hsv_s=0.5, hsv_v=0.4)
- Cùng 2 model: Baseline YOLO11n-seg và YOLO11n-seg + SimAM + CA
- Cùng evaluation pipeline (healthy-aware score, disease miss rate, FP rate)

**Noise Testing — 11 điều kiện (mở rộng từ v2):**
| # | Noise | Mô phỏng |
|---|-------|----------|
| N01 | Gaussian σ=20 | Camera sensor noise |
| N02 | Salt & Pepper p=0.03 | Dead pixel / sensor noise |
| N03 | JPEG Artifact q=15 | Ảnh nén qua Zalo/Messenger |
| N04 | Blue Color Cast +40 | Màu nước ao (blue) |
| N05 | Low Contrast ×0.5 | Toàn ảnh kém tương phản |
| N06 | Gaussian Blur σ=2.5 | Tôm mờ / không lấy nét |
| N07 | Heavy Erasing | Che khuất mật độ cao (tôm chồng nhau) |
| N08 | Motion Blur kernel=11 | Tôm di chuyển / camera rung |
| N09 | Overexposure +40% | Đèn UV ao / flash chói |
| N10 | Specular Glare ×3 | Phản chiếu mặt nước / vỏ tôm ướt |
| N11 | Turbidity (nước đục) | Ao sau mưa, tảo nở rộ |


In [ ]:
# Verify T4 GPU — Runtime > Change runtime type > GPU > T4
import subprocess
out = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(out.stdout if out.returncode == 0 else
      'No GPU detected. Enable T4: Runtime > Change runtime type > T4 GPU')


## 1. Download dataset

In [ ]:
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec('roboflow') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'roboflow'])

from roboflow import Roboflow

ROBOFLOW_API_KEY_DIRECT = 'KOEk0qLzBFDc7zfyxtgs'
ROBOFLOW_WORKSPACE = 'lets-try-this'
ROBOFLOW_PROJECT = 'shrimpdishandsegv2'
ROBOFLOW_VERSION = 1
ROBOFLOW_FORMAT = 'yolo26'


def get_roboflow_api_key():
    if ROBOFLOW_API_KEY_DIRECT.strip():
        return ROBOFLOW_API_KEY_DIRECT.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
        if key:
            return key
    except Exception:
        pass
    return os.environ.get('ROBOFLOW_API_KEY', '').strip()


api_key = get_roboflow_api_key()
if not api_key:
    raise RuntimeError(
        'Missing Roboflow API key. Add a Kaggle Secret named ROBOFLOW_API_KEY, '
        'set the ROBOFLOW_API_KEY environment variable, or temporarily fill ROBOFLOW_API_KEY_DIRECT.'
    )

rf = Roboflow(api_key=api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download(ROBOFLOW_FORMAT)


## 2. Grouped-stratified split to prevent leakage

In [ ]:
import os
import random
import re
import shutil
from collections import defaultdict, Counter
from pathlib import Path

SEED = 42
random.seed(SEED)

base_path = '/kaggle/working/shrimpDisHandSegV2-1'
train_path = os.path.join(base_path, 'train')
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

# Prevent leakage from multiple photos of the same shrimp.
# Expected original filename: <diseasename>-<shrimpid>-img-<imgnum>.jpg
# Roboflow may export names like <diseasename>-<shrimpid>-img-<imgnum>_jpg.rf.<hash>.jpg.
# Example disease names: Healthy, BG, WSSV_BG, WSSV.
GROUP_SPLIT_BY_SHRIMP = True
GROUP_STRATIFY_BY_DISEASE = True
REBUILD_SPLIT_FROM_ALL_SPLITS = True
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10

SHRIMP_NAME_PATTERN = re.compile(
    r'^(?P<disease>Healthy|BG|WSSV_BG|WSSV)-(?P<shrimp_id>.+)-img-(?P<img_num>\d+)$',
    re.IGNORECASE,
)


def normalize_roboflow_stem(stem):
    """Recover the original filename stem from Roboflow-exported names."""
    stem = re.sub(r'_(jpg|jpeg|png|bmp|webp)\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    stem = re.sub(r'\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    return stem


for split in ['train', 'valid', 'test']:
    for sub in ['images', 'labels']:
        os.makedirs(os.path.join(base_path, split, sub), exist_ok=True)


def parse_shrimp_group_key(image_name):
    """Return a stable group key so all images from one shrimp stay in one split."""
    stem = normalize_roboflow_stem(Path(image_name).stem)
    match = SHRIMP_NAME_PATTERN.match(stem)
    if not match:
        return f'unparsed::{Path(image_name).stem}', 'unparsed', None, None

    disease = match.group('disease')
    shrimp_id = match.group('shrimp_id')
    img_num = int(match.group('img_num'))
    group_key = f'{disease.lower()}::{shrimp_id}'
    return group_key, disease, shrimp_id, img_num


def image_files_in_split(split):
    image_dir = Path(base_path) / split / 'images'
    return sorted(
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )


def move_image_and_label(image_path, target_split):
    target_img_dir = Path(base_path) / target_split / 'images'
    target_lbl_dir = Path(base_path) / target_split / 'labels'
    target_img_dir.mkdir(parents=True, exist_ok=True)
    target_lbl_dir.mkdir(parents=True, exist_ok=True)

    label_name = f'{image_path.stem}.txt'
    label_src = image_path.parent.parent / 'labels' / label_name
    image_dst = target_img_dir / image_path.name
    label_dst = target_lbl_dir / label_name

    if image_path.resolve() != image_dst.resolve():
        if image_dst.exists():
            raise FileExistsError(f'Duplicate image destination would be overwritten: {image_dst}')
        shutil.move(str(image_path), str(image_dst))

    if label_src.exists():
        if label_src.resolve() != label_dst.resolve():
            if label_dst.exists():
                raise FileExistsError(f'Duplicate label destination would be overwritten: {label_dst}')
            shutil.move(str(label_src), str(label_dst))
    else:
        label_dst.write_text('')


def rebuild_train_pool_from_all_splits():
    all_images = []
    for split in ['train', 'valid', 'test']:
        all_images.extend(image_files_in_split(split))

    for image_path in sorted(all_images):
        move_image_and_label(image_path, 'train')

    return image_files_in_split('train')


def remove_yolo_label_caches(root):
    for cache_path in Path(root).glob('**/*.cache'):
        cache_path.unlink()
        print(f'Removed stale cache: {cache_path}')


def disease_for_group(filenames):
    diseases = []
    for filename in filenames:
        _, disease, _, _ = parse_shrimp_group_key(filename)
        diseases.append(disease)
    counts = Counter(diseases)
    if len(counts) > 1:
        print(f'Warning: group has mixed disease names: {dict(counts)}')
    return counts.most_common(1)[0][0]


def split_one_stratum(items):
    n = len(items)
    train_count = int(TRAIN_RATIO * n)
    val_count = int(VAL_RATIO * n)
    test_count = n - train_count - val_count

    if n >= 3:
        if val_count == 0:
            val_count = 1
            train_count -= 1
        if test_count == 0:
            test_count = 1
            train_count -= 1
    if train_count < 1 and n > 0:
        train_count = 1
    while train_count + val_count + test_count > n:
        train_count -= 1
    test_count = n - train_count - val_count

    return (
        items[:train_count],
        items[train_count:train_count + val_count],
        items[train_count + val_count:],
    )


def grouped_stratified_split(group_items):
    strata = defaultdict(list)
    for group_key, filenames in group_items:
        strata[disease_for_group(filenames)].append((group_key, filenames))

    split_to_groups = {'train': [], 'valid': [], 'test': []}
    rng = random.Random(SEED)
    for disease, items in sorted(strata.items()):
        items = sorted(items, key=lambda item: item[0])
        rng.shuffle(items)
        train_items, val_items, test_items = split_one_stratum(items)
        split_to_groups['train'].extend(train_items)
        split_to_groups['valid'].extend(val_items)
        split_to_groups['test'].extend(test_items)
        print(
            f'  - {disease}: {len(train_items)} train groups, '
            f'{len(val_items)} valid groups, {len(test_items)} test groups'
        )

    for split in split_to_groups:
        split_to_groups[split] = sorted(split_to_groups[split], key=lambda item: item[0])
    return split_to_groups


def grouped_random_split(group_items):
    group_items = sorted(group_items, key=lambda item: item[0])
    random.Random(SEED).shuffle(group_items)
    n_groups = len(group_items)
    train_group_count = int(TRAIN_RATIO * n_groups)
    val_group_count = int(VAL_RATIO * n_groups)
    return {
        'train': group_items[:train_group_count],
        'valid': group_items[train_group_count:train_group_count + val_group_count],
        'test': group_items[train_group_count + val_group_count:],
    }


def split_summary(split_groups):
    group_diseases = Counter()
    image_diseases = Counter()
    for _, filenames in split_groups:
        group_diseases[disease_for_group(filenames)] += 1
        for filename in filenames:
            _, disease, _, _ = parse_shrimp_group_key(filename)
            image_diseases[disease] += 1
    return group_diseases, image_diseases


def split_grouped_by_shrimp():
    if REBUILD_SPLIT_FROM_ALL_SPLITS:
        image_paths = rebuild_train_pool_from_all_splits()
    else:
        image_paths = image_files_in_split('train')

    groups = defaultdict(list)
    disease_counts = Counter()
    unparsed = []

    for image_path in image_paths:
        group_key, disease, shrimp_id, img_num = parse_shrimp_group_key(image_path.name)
        groups[group_key].append(image_path.name)
        disease_counts[disease] += 1
        if disease == 'unparsed':
            unparsed.append(image_path.name)

    group_items = sorted(groups.items(), key=lambda item: item[0])
    if GROUP_STRATIFY_BY_DISEASE:
        print('Building shrimp-grouped, disease-stratified split:')
        split_to_groups = grouped_stratified_split(group_items)
    else:
        print('Building shrimp-grouped random split:')
        split_to_groups = grouped_random_split(group_items)

    for split, split_groups in split_to_groups.items():
        for _, filenames in split_groups:
            for filename in filenames:
                move_image_and_label(Path(base_path) / 'train' / 'images' / filename, split)

    print('Shrimp-grouped split complete:')
    for split, split_groups in split_to_groups.items():
        image_count = sum(len(filenames) for _, filenames in split_groups)
        group_diseases, image_diseases = split_summary(split_groups)
        print(f'  - {split}: {len(split_groups)} shrimp groups, {image_count} images')
        print(f'    group disease counts: {dict(sorted(group_diseases.items()))}')
        print(f'    image disease counts: {dict(sorted(image_diseases.items()))}')

    print('Source filename disease counts before split:', dict(sorted(disease_counts.items())))
    if unparsed:
        print(f'Warning: {len(unparsed)} filenames did not match the shrimp naming pattern. They were split as single-image groups.')
        print('First unparsed examples:', unparsed[:10])

    group_to_split = {}
    leakage = []
    for split in ['train', 'valid', 'test']:
        for image_path in image_files_in_split(split):
            group_key, *_ = parse_shrimp_group_key(image_path.name)
            previous_split = group_to_split.setdefault(group_key, split)
            if previous_split != split:
                leakage.append((group_key, previous_split, split, image_path.name))

    if leakage:
        raise RuntimeError(f'Shrimp-level split leakage detected: {leakage[:10]}')
    print('Shrimp-level leakage check passed.')
    remove_yolo_label_caches(base_path)


if GROUP_SPLIT_BY_SHRIMP:
    split_grouped_by_shrimp()
else:
    valid_images_dir = Path(base_path) / 'valid' / 'images'
    test_images_dir = Path(base_path) / 'test' / 'images'

    if not any(valid_images_dir.glob('*')) and not any(test_images_dir.glob('*')):
        image_files = sorted(
            f for f in os.listdir(os.path.join(train_path, 'images'))
            if f.lower().endswith(IMAGE_EXTENSIONS)
        )
        random.shuffle(image_files)

        train_count = int(0.8 * len(image_files))
        val_count = int(0.1 * len(image_files))
        val_files = image_files[train_count:train_count + val_count]
        test_files = image_files[train_count + val_count:]

        def move_files(files, target_split):
            for f in files:
                move_image_and_label(Path(train_path) / 'images' / f, target_split)

        move_files(val_files, 'valid')
        move_files(test_files, 'test')
        print(f"Image-level split complete: {len(image_files) - len(val_files) - len(test_files)} train, {len(val_files)} val, {len(test_files)} test")
        remove_yolo_label_caches(base_path)
    else:
        print('Existing valid/test split detected. Keeping downloaded split.')


## Setup Ultralytics, YOLO model, and dataset paths

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

from ultralytics import YOLO
import os
from pathlib import Path

base_path = "/kaggle/working/shrimpDisHandSegV2-1"
data_yaml_path = os.path.join(base_path, "data.yaml")

# Keep the model name and run name tied together so reports are not mislabeled.
# This baseline model is also used as pretrained initialization for attention models.
YOLO_MODELS = [
    # "yolov8n-seg.pt",
    # "yolov8m-seg.pt",
    "yolo11n-seg.pt",
    # "yolo11m-seg.pt",
    # "yolo26n-seg.pt",
    # "yolo26m-seg.pt",
]

YOLO_MODEL = YOLO_MODELS[0]
MODEL_STEM = Path(YOLO_MODEL).stem
RUN_BASE_NAME = f"{MODEL_STEM}_shrimp_seg_clean_baseline"

# Load once here as a smoke check. Training cells instantiate fresh models per experiment.
model = YOLO(YOLO_MODEL)

print("Configured segmentation models:")
for configured_model in YOLO_MODELS:
    print(f"  - {configured_model}")
print(f"Run name prefix: {RUN_BASE_NAME}")
print(f"base_path: {base_path}")
print(f"data_yaml_path: {data_yaml_path}")


## 3. Update data.yaml to absolute paths

In [ ]:
from pathlib import Path
import yaml

# data_yaml_path and base_path are defined in the setup cell above.
if "base_path" not in globals():
    base_path = "/kaggle/working/shrimpDisHandSegV2-1"

if "data_yaml_path" not in globals():
    data_yaml_path = str(Path(base_path) / "data.yaml")

if not Path(data_yaml_path).exists():
    raise FileNotFoundError(f"data.yaml not found: {data_yaml_path}")

# Update data.yaml to use correct absolute paths after grouped-stratified split.
with open(data_yaml_path, "r") as f:
    content = yaml.safe_load(f)

content["train"] = str(Path(base_path) / "train" / "images")
content["val"] = str(Path(base_path) / "valid" / "images")
content["test"] = str(Path(base_path) / "test" / "images")

with open(data_yaml_path, "w") as f:
    yaml.safe_dump(content, f, sort_keys=False)

print("data.yaml updated with absolute paths.")
print("data_yaml_path:", data_yaml_path)
print("train:", content["train"])
print("val:", content["val"])
print("test:", content["test"])


## 4. EDA

In [ ]:
import os
import yaml
import matplotlib.pyplot as plt
import cv2
import numpy as np
from collections import Counter
from pathlib import Path

# Load class names from data.yaml. Empty label files are healthy shrimp negatives.
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

class_names = data_config.get('names', [])
HEALTHY_CLASS_NAME = 'healthy'
print(f"Disease mask classes found: {class_names}")
print(f"Empty label files will be treated as: {HEALTHY_CLASS_NAME} shrimp negatives")


def split_label_stats(label_dir):
    instance_counts = Counter()
    labeled_images = 0
    healthy_images = 0
    missing_or_empty = 0
    label_dir = Path(label_dir)
    for label_file in label_dir.glob('*.txt'):
        lines = [line.strip() for line in label_file.read_text().splitlines() if line.strip()]
        if not lines:
            healthy_images += 1
            continue
        labeled_images += 1
        for line in lines:
            class_id = int(float(line.split()[0]))
            instance_counts[class_id] += 1
    return {
        'instance_counts': instance_counts,
        'labeled_images': labeled_images,
        'healthy_images': healthy_images,
        'total_label_files': labeled_images + healthy_images,
    }


stats = {}
for split in ['train', 'valid', 'test']:
    label_dir = os.path.join(base_path, split, 'labels')
    stats[split] = split_label_stats(label_dir)

for split, split_stats in stats.items():
    print(f"\n{split.capitalize()} Split:")
    print(f"  - labeled disease images: {split_stats['labeled_images']}")
    print(f"  - healthy negative images: {split_stats['healthy_images']}")
    for cid, count in split_stats['instance_counts'].items():
        name = class_names[cid] if cid < len(class_names) else f"Unknown({cid})"
        print(f"  - {name}: {count} mask instances")

## 5. Class imbalance visualization

In [ ]:
import pandas as pd
import seaborn as sns

instance_plot_data = []
image_plot_data = []
for split, split_stats in stats.items():
    image_plot_data.append({'Split': split, 'Class': HEALTHY_CLASS_NAME, 'Images': split_stats['healthy_images']})
    image_plot_data.append({'Split': split, 'Class': 'diseased_labeled', 'Images': split_stats['labeled_images']})
    for cid, count in split_stats['instance_counts'].items():
        instance_plot_data.append({'Split': split, 'Class': class_names[cid], 'Instances': count})

if instance_plot_data:
    df_instances = pd.DataFrame(instance_plot_data)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_instances, x='Split', y='Instances', hue='Class')
    plt.title('Disease Mask Instance Distribution across Splits')
    plt.show()

if image_plot_data:
    df_images = pd.DataFrame(image_plot_data)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_images, x='Split', y='Images', hue='Class')
    plt.title('Healthy Negative vs Diseased-Labeled Image Counts')
    plt.show()

for split, split_stats in stats.items():
    total_instances = sum(split_stats['instance_counts'].values())
    blackgill_ratio = (split_stats['instance_counts'].get(0, 0) / max(1, total_instances)) * 100
    healthy_ratio = (split_stats['healthy_images'] / max(1, split_stats['total_label_files'])) * 100
    print(f"{split.capitalize()}: {blackgill_ratio:.2f}% blackgill instances; {healthy_ratio:.2f}% healthy negative images")

## 6. Setup Ultralytics


In [ ]:
from pathlib import Path
import subprocess
import sys

ULTRA_DIR = Path("/kaggle/working/ultralytics")

if not ULTRA_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/ultralytics/ultralytics.git", str(ULTRA_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ULTRA_DIR)], check=True)

sys.path.insert(0, str(ULTRA_DIR))

for module_name in list(sys.modules.keys()):
    if module_name == "ultralytics" or module_name.startswith("ultralytics."):
        del sys.modules[module_name]

import ultralytics
from ultralytics import YOLO

print("Using Ultralytics from:", ultralytics.__file__)
assert str(ULTRA_DIR) in ultralytics.__file__
print("Ultralytics setup complete (baseline-only, no attention patches).")


In [ ]:
# Ultralytics already installed in previous cell.
pass


## 7. (Skipped — no attention YAML needed for baseline-only run)


In [ ]:
# No attention YAML files needed for baseline-only run.
MODEL_YAML_PATHS = {}
print("Baseline-only run — no attention YAML configs created.")


## 8. Select experiment group

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "/kaggle/working/ultralytics")

EXPERIMENT_ROOT_STR = "/kaggle/working/shrimp_yolo_seg_baseline_noise_ablation"
RUN_BASE_NAME = "yolo11n-seg_shrimp_seg_baseline_noise_ablation"

EXPERIMENTS = [
    {
        "key": "baseline",
        "name": "Baseline YOLO11n-seg",
        "model_type": "baseline",
    },
]

print("Experiments:")
for e in EXPERIMENTS:
    print("-", e["key"], ":", e["name"])


## 9. Train baseline model and build summary

In [ ]:
import csv
import gc
import math
import os
import random
import shutil
import time
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from ultralytics import YOLO

# =========================================================
# Reproducibility
# =========================================================
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as e:
    print("Deterministic warning:", e)

# =========================================================
# Paths and clean augmentation
# =========================================================
EXPERIMENT_ROOT = Path(EXPERIMENT_ROOT_STR)
RUNS_DIR = Path("/kaggle/working/runs/segment")
REPORT_DIR = EXPERIMENT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

COUNT_PENALTY_WEIGHT = 0.05
DISEASE_MISS_PENALTY_WEIGHT = 0.15
HEALTHY_FP_PENALTY_WEIGHT = 0.10
PREDICT_CONF_FOR_COUNT = 0.25

# ── Baseline: light augmentation — identical to aip491-01 baseline notebook ──
# (mosaic=0, erasing=0, flipud=0, low hsv/scale/translate)
BASELINE_TRAIN_ARGS = {
    "auto_augment": None,
    "erasing": 0.0,
    "mosaic": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "fliplr": 0.5,
    "flipud": 0.0,
    "hsv_h": 0.01,
    "hsv_s": 0.35,
    "hsv_v": 0.20,
    "degrees": 0.0,
    "translate": 0.05,
    "scale": 0.20,
    "shear": 0.0,
    "perspective": 0.0,
    "multi_scale": 0.0,
    "bgr": 0.0,
}


IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
base_path = "/kaggle/working/shrimpDisHandSegV2-1"
data_yaml_path = os.path.join(base_path, "data.yaml")

# =========================================================
# Disable default Ultralytics Albumentations hook if available
# =========================================================
try:
    import ultralytics.data.augment as yolo_aug
    if hasattr(yolo_aug, "Albumentations"):
        class NoAlbumentations:
            def __init__(self, *args, **kwargs):
                pass
            def __call__(self, labels):
                return labels
        yolo_aug.Albumentations = NoAlbumentations
        print("Disabled default Ultralytics Albumentations hook.")
except Exception as e:
    print("Could not disable Albumentations hook:", e)

# =========================================================
# Utility functions
# =========================================================
def remove_yolo_label_caches(root):
    for cache_path in Path(root).glob("**/*.cache"):
        cache_path.unlink()
        print(f"Removed stale cache: {cache_path}")

def count_labeled_images(label_dir):
    labeled = 0
    healthy = 0
    instances = 0
    for label_path in Path(label_dir).glob("*.txt"):
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        if lines:
            labeled += 1
            instances += len(lines)
        else:
            healthy += 1
    return {"labeled_images": labeled, "healthy_images": healthy, "instances": instances}

def write_data_yaml(dataset_dir, yaml_path, val_dir="valid", test_dir="test"):
    with open(data_yaml_path, "r") as f:
        content = yaml.safe_load(f)
    content["train"] = str(Path(dataset_dir) / "train" / "images")
    content["val"] = str(Path(dataset_dir) / val_dir / "images")
    content["test"] = str(Path(dataset_dir) / test_dir / "images")
    with open(yaml_path, "w") as f:
        yaml.safe_dump(content, f, sort_keys=False)
    return yaml_path

def copy_dataset_for_experiment(exp_key):
    src = Path(base_path)
    dst = EXPERIMENT_ROOT / exp_key / "dataset"
    if dst.exists():
        shutil.rmtree(dst)
    ignore = shutil.ignore_patterns("runs", "*.cache", ".clahe_applied")
    shutil.copytree(src, dst, ignore=ignore)
    remove_yolo_label_caches(dst)
    return dst

def find_image_for_label(image_dir, label_name):
    stem = Path(label_name).stem
    for ext in IMAGE_EXTENSIONS:
        candidate = Path(image_dir) / f"{stem}{ext}"
        if candidate.exists():
            return candidate
    return None

def copy_split_by_label_state(src_dataset, dst_dataset, split, want_labeled):
    src_images = Path(src_dataset) / split / "images"
    src_labels = Path(src_dataset) / split / "labels"
    dst_images = Path(dst_dataset) / split / "images"
    dst_labels = Path(dst_dataset) / split / "labels"
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)
    copied = 0
    for label_path in sorted(src_labels.glob("*.txt")):
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        is_labeled = bool(lines)
        if is_labeled != want_labeled:
            continue
        image_path = find_image_for_label(src_images, label_path.name)
        if image_path is None:
            continue
        shutil.copy2(image_path, dst_images / image_path.name)
        shutil.copy2(label_path, dst_labels / label_path.name)
        copied += 1
    return copied

def make_state_eval_dataset(src_dataset, exp_key, state_name, want_labeled):
    dst = EXPERIMENT_ROOT / exp_key / f"dataset_{state_name}_eval"
    if dst.exists():
        shutil.rmtree(dst)
    for sub in ["images", "labels"]:
        (dst / "train" / sub).mkdir(parents=True, exist_ok=True)
    copied = {}
    for split in ["valid", "test"]:
        copied[split] = copy_split_by_label_state(src_dataset, dst, split, want_labeled=want_labeled)
    yaml_path = dst / f"data_{state_name}.yaml"
    write_data_yaml(dst, yaml_path)
    print(f"{state_name} eval dataset for {exp_key}: {copied}")
    return dst, yaml_path, copied

def make_labeled_only_eval_dataset(src_dataset, exp_key):
    return make_state_eval_dataset(src_dataset, exp_key, "labeled_only", want_labeled=True)

def make_healthy_only_eval_dataset(src_dataset, exp_key):
    return make_state_eval_dataset(src_dataset, exp_key, "healthy_only", want_labeled=False)

def metric_value(metrics, dotted_path, default=float("nan")):
    obj = metrics
    for part in dotted_path.split("."):
        if not hasattr(obj, part):
            return default
        obj = getattr(obj, part)
    try:
        return float(obj)
    except Exception:
        return default

def list_images(image_dir):
    image_dir = Path(image_dir)
    images = []
    for ext in IMAGE_EXTENSIONS:
        images.extend(image_dir.rglob(f"*{ext}"))
    return sorted(images)

def image_to_label_path(image_path):
    return Path(image_path).parent.parent / "labels" / f"{Path(image_path).stem}.txt"

def read_label_instances(label_path):
    label_path = Path(label_path)
    if not label_path.exists():
        return []
    rows = []
    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            cls = int(float(parts[0]))
            nums = [float(x) for x in parts[1:]]
        except Exception:
            continue
        rows.append((cls, nums))
    return rows

def instance_to_box(instance, image_path):
    cls, nums = instance
    w, h = Image.open(image_path).size
    if len(nums) == 4:
        xc, yc, bw, bh = nums
        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h
        x2 = (xc + bw / 2) * w
        y2 = (yc + bh / 2) * h
    else:
        coords = np.array(nums, dtype=float).reshape(-1, 2)
        xs = coords[:, 0] * w
        ys = coords[:, 1] * h
        x1, y1, x2, y2 = xs.min(), ys.min(), xs.max(), ys.max()
    return [float(x1), float(y1), float(x2), float(y2), int(cls)]

def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a[:4]
    bx1, by1, bx2, by2 = b[:4]
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def get_class_names(yaml_path):
    with open(yaml_path, "r") as f:
        cfg = yaml.safe_load(f)
    names = cfg.get("names", {})
    if isinstance(names, list):
        return {i: n for i, n in enumerate(names)}
    if isinstance(names, dict):
        return {int(k): v for k, v in names.items()}
    return {}

def count_prediction_errors(model, images_dir, labels_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = list_images(images_dir)
    if not image_paths:
        return {
            "images": 0, "gt_total": 0, "pred_box_total": 0, "pred_mask_total": 0,
            "box_count_mae": float("nan"), "mask_count_mae": float("nan"),
            "box_count_exact": float("nan"), "mask_count_exact": float("nan"),
            "disease_images": 0, "disease_box_miss_images": 0, "disease_mask_miss_images": 0,
            "disease_box_miss_rate": float("nan"), "disease_mask_miss_rate": float("nan")
        }
    results = model.predict(source=[str(p) for p in image_paths], imgsz=640, conf=conf, verbose=False)
    box_errors, mask_errors, box_exact, mask_exact = [], [], [], []
    gt_total = pred_box_total = pred_mask_total = 0
    disease_images = disease_box_miss_images = disease_mask_miss_images = 0
    for image_path, result in zip(image_paths, results):
        label_path = Path(labels_dir) / f"{image_path.stem}.txt"
        gt_count = len([line for line in label_path.read_text().splitlines() if line.strip()]) if label_path.exists() else 0
        box_count = len(result.boxes) if result.boxes is not None else 0
        mask_count = len(result.masks) if result.masks is not None else 0
        denom = max(1, gt_count)
        box_errors.append(abs(box_count - gt_count) / denom)
        mask_errors.append(abs(mask_count - gt_count) / denom)
        box_exact.append(float(box_count == gt_count))
        mask_exact.append(float(mask_count == gt_count))
        if gt_count > 0:
            disease_images += 1
            disease_box_miss_images += int(box_count == 0)
            disease_mask_miss_images += int(mask_count == 0)
        gt_total += gt_count
        pred_box_total += box_count
        pred_mask_total += mask_count
    return {
        "images": len(image_paths), "gt_total": gt_total,
        "pred_box_total": pred_box_total, "pred_mask_total": pred_mask_total,
        "box_count_mae": sum(box_errors) / len(box_errors),
        "mask_count_mae": sum(mask_errors) / len(mask_errors),
        "box_count_exact": sum(box_exact) / len(box_exact),
        "mask_count_exact": sum(mask_exact) / len(mask_exact),
        "disease_images": disease_images,
        "disease_box_miss_images": disease_box_miss_images,
        "disease_mask_miss_images": disease_mask_miss_images,
        "disease_box_miss_rate": disease_box_miss_images / disease_images if disease_images else float("nan"),
        "disease_mask_miss_rate": disease_mask_miss_images / disease_images if disease_images else float("nan"),
    }

def healthy_false_positive_summary(model, images_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = list_images(images_dir)
    if not image_paths:
        return {
            "healthy_images": 0, "healthy_images_with_box_fp": 0, "healthy_images_with_mask_fp": 0,
            "healthy_box_fp_rate": float("nan"), "healthy_mask_fp_rate": float("nan"),
            "healthy_fp_boxes_total": 0, "healthy_fp_masks_total": 0,
            "healthy_fp_boxes_per_image": float("nan"), "healthy_fp_masks_per_image": float("nan"),
            "healthy_avg_fp_confidence": float("nan"),
        }
    results = model.predict(source=[str(p) for p in image_paths], imgsz=640, conf=conf, verbose=False)
    images_with_box_fp = images_with_mask_fp = box_total = mask_total = 0
    confidences = []
    for result in results:
        box_count = len(result.boxes) if result.boxes is not None else 0
        mask_count = len(result.masks) if result.masks is not None else 0
        if box_count > 0:
            images_with_box_fp += 1
            try:
                confidences.extend([float(v) for v in result.boxes.conf.detach().cpu().tolist()])
            except Exception:
                pass
        if mask_count > 0:
            images_with_mask_fp += 1
        box_total += box_count
        mask_total += mask_count
    n = len(image_paths)
    return {
        "healthy_images": n,
        "healthy_images_with_box_fp": images_with_box_fp,
        "healthy_images_with_mask_fp": images_with_mask_fp,
        "healthy_box_fp_rate": images_with_box_fp / n,
        "healthy_mask_fp_rate": images_with_mask_fp / n,
        "healthy_fp_boxes_total": box_total,
        "healthy_fp_masks_total": mask_total,
        "healthy_fp_boxes_per_image": box_total / n,
        "healthy_fp_masks_per_image": mask_total / n,
        "healthy_avg_fp_confidence": sum(confidences) / len(confidences) if confidences else 0.0,
    }

def read_best_epoch_from_results(run_path):
    results_csv = Path(run_path) / "results.csv"
    if not results_csv.exists():
        return {}
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    mask_col = "metrics/mAP50(M)"
    if mask_col not in df.columns:
        return {"epochs_ran": len(df)}
    best_idx = df[mask_col].idxmax()
    first = df.iloc[0]
    best = df.iloc[best_idx]
    last = df.iloc[-1]
    return {
        "epochs_ran": int(len(df)),
        "best_epoch_by_mask_map50": int(best["epoch"]) if "epoch" in df.columns else int(best_idx + 1),
        "first_train_seg_loss": float(first.get("train/seg_loss", float("nan"))),
        "best_val_mask_map50": float(best.get(mask_col, float("nan"))),
        "best_val_mask_map50_95": float(best.get("metrics/mAP50-95(M)", float("nan"))),
        "last_val_mask_map50": float(last.get(mask_col, float("nan"))),
        "last_val_mask_map50_95": float(last.get("metrics/mAP50-95(M)", float("nan"))),
        "last_train_seg_loss": float(last.get("train/seg_loss", float("nan"))),
        "last_val_seg_loss": float(last.get("val/seg_loss", float("nan"))),
        "seg_loss_gap_val_minus_train": float(last.get("val/seg_loss", float("nan")) - last.get("train/seg_loss", float("nan"))),
    }

def class_level_test_analysis(model, dataset_dir, yaml_path, report_dir, conf=0.25, iou_threshold=0.50):
    report_dir = Path(report_dir)
    report_dir.mkdir(parents=True, exist_ok=True)
    class_names = get_class_names(yaml_path)
    def cname(cid): return class_names.get(int(cid), str(cid))

    # Annotation count by class
    annotation_rows = []
    for split in ["train", "valid", "test"]:
        image_dir = Path(dataset_dir) / split / "images"
        instance_counter, image_counter = Counter(), Counter()
        for img in list_images(image_dir):
            instances = read_label_instances(image_to_label_path(img))
            seen = set()
            for inst in instances:
                cls = inst[0]
                instance_counter[cls] += 1
                seen.add(cls)
            for cls in seen:
                image_counter[cls] += 1
        for cls, count in sorted(instance_counter.items()):
            annotation_rows.append({
                "split": split,
                "class_id": cls,
                "class_name": cname(cls),
                "annotation_count": count,
                "image_count_with_class": image_counter[cls],
            })
    annotation_df = pd.DataFrame(annotation_rows)
    annotation_df.to_csv(report_dir / "annotation_count_by_class.csv", index=False)

    if not annotation_df.empty:
        pivot = annotation_df.pivot_table(index="class_name", columns="split", values="annotation_count", aggfunc="sum", fill_value=0)
        ax = pivot.plot(kind="bar", figsize=(10, 5))
        ax.set_title("Annotation count by class and split")
        ax.set_xlabel("Class")
        ax.set_ylabel("Annotation count")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.savefig(report_dir / "annotation_count_by_class.png", dpi=200)
        plt.close()

    # TEST confusion/miss/FP
    test_images = list_images(Path(dataset_dir) / "test" / "images")
    confusion_counter, miss_counter, fp_counter, matched_counter = Counter(), Counter(), Counter(), Counter()
    for img in test_images:
        gt_instances = read_label_instances(image_to_label_path(img))
        gt_boxes = [instance_to_box(inst, img) for inst in gt_instances]
        result = model.predict(str(img), imgsz=640, conf=conf, iou=0.5, verbose=False)[0]
        pred_boxes = []
        if result.boxes is not None and len(result.boxes) > 0:
            xyxy = result.boxes.xyxy.cpu().numpy()
            cls_arr = result.boxes.cls.cpu().numpy().astype(int)
            conf_arr = result.boxes.conf.cpu().numpy()
            for box, cls, score in zip(xyxy, cls_arr, conf_arr):
                pred_boxes.append([float(box[0]), float(box[1]), float(box[2]), float(box[3]), int(cls), float(score)])
        used_pred = set()
        for gt in gt_boxes:
            best_iou, best_j = 0.0, -1
            for j, pred in enumerate(pred_boxes):
                if j in used_pred:
                    continue
                iou = box_iou(gt, pred)
                if iou > best_iou:
                    best_iou, best_j = iou, j
            gt_cls = int(gt[4])
            if best_j >= 0 and best_iou >= iou_threshold:
                used_pred.add(best_j)
                pred_cls = int(pred_boxes[best_j][4])
                matched_counter[cname(gt_cls)] += 1
                if pred_cls != gt_cls:
                    confusion_counter[(cname(gt_cls), cname(pred_cls))] += 1
            else:
                miss_counter[cname(gt_cls)] += 1
        for j, pred in enumerate(pred_boxes):
            if j not in used_pred:
                fp_counter[cname(int(pred[4]))] += 1

    confusion_df = pd.DataFrame(
        [{"gt_class": gt, "pred_class": pred, "confused_count": count} for (gt, pred), count in confusion_counter.items()]
    ).sort_values("confused_count", ascending=False) if confusion_counter else pd.DataFrame(columns=["gt_class", "pred_class", "confused_count"])

    miss_df = pd.DataFrame(
        [{"class_name": cls, "missed_box_count": count} for cls, count in miss_counter.items()]
    ).sort_values("missed_box_count", ascending=False) if miss_counter else pd.DataFrame(columns=["class_name", "missed_box_count"])

    fp_df = pd.DataFrame(
        [{"class_name": cls, "false_positive_count": count} for cls, count in fp_counter.items()]
    ).sort_values("false_positive_count", ascending=False) if fp_counter else pd.DataFrame(columns=["class_name", "false_positive_count"])

    confusion_df.to_csv(report_dir / "test_class_confusion_counts.csv", index=False)
    miss_df.to_csv(report_dir / "test_missed_box_counts_by_class.csv", index=False)
    fp_df.to_csv(report_dir / "test_false_positive_counts_by_class.csv", index=False)

    if not confusion_df.empty:
        labels = confusion_df.apply(lambda r: f"{r['gt_class']} → {r['pred_class']}", axis=1)
        plt.figure(figsize=(10, max(4, 0.5 * len(labels))))
        plt.barh(labels, confusion_df["confused_count"])
        plt.title("TEST class confusion counts")
        plt.xlabel("Count")
        plt.ylabel("GT → Pred")
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.savefig(report_dir / "test_class_confusion_counts.png", dpi=200)
        plt.close()

    if not miss_df.empty:
        plt.figure(figsize=(8, 4))
        plt.bar(miss_df["class_name"], miss_df["missed_box_count"])
        plt.title("TEST missed boxes by class")
        plt.xlabel("Class")
        plt.ylabel("Missed box count")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.savefig(report_dir / "test_missed_boxes_by_class.png", dpi=200)
        plt.close()

    return {
        "test_confused_total": int(confusion_df["confused_count"].sum()) if not confusion_df.empty else 0,
        "test_missed_box_total": int(miss_df["missed_box_count"].sum()) if not miss_df.empty else 0,
        "test_false_positive_total": int(fp_df["false_positive_count"].sum()) if not fp_df.empty else 0,
    }

def save_loss_and_correlation_plots(run_path, report_dir):
    report_dir = Path(report_dir)
    report_dir.mkdir(parents=True, exist_ok=True)
    results_csv = Path(run_path) / "results.csv"
    if not results_csv.exists():
        return {}
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    df.to_csv(report_dir / "results_clean_columns.csv", index=False)

    loss_cols = [c for c in df.columns if "loss" in c.lower()]
    if loss_cols:
        plt.figure(figsize=(12, 6))
        x = df["epoch"] if "epoch" in df.columns else df.index
        for col in loss_cols:
            plt.plot(x, df[col], label=col)
        plt.title("Train / validation loss curves")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(report_dir / "train_val_loss_curves.png", dpi=200)
        plt.close()

    numeric_df = df.select_dtypes(include=[np.number])
    if numeric_df.shape[1] >= 2:
        corr = numeric_df.corr()
        corr.to_csv(report_dir / "training_metrics_correlation_matrix.csv")
        plt.figure(figsize=(12, 10))
        im = plt.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
        plt.colorbar(im, fraction=0.046, pad=0.04)
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
        plt.yticks(range(len(corr.columns)), corr.columns)
        plt.title("Correlation matrix of training metrics")
        plt.tight_layout()
        plt.savefig(report_dir / "training_metrics_correlation_matrix.png", dpi=200)
        plt.close()
    return {"results_csv": str(results_csv)}

def make_model(exp):
    return YOLO("yolo11n-seg.pt")

def run_experiment(exp):
    print("\n" + "#" * 90)
    print(f"Starting experiment: {exp['name']}")
    print("#" * 90)

    dataset_dir = copy_dataset_for_experiment(exp["key"])
    remove_yolo_label_caches(dataset_dir)
    yaml_path = dataset_dir / "data.yaml"
    write_data_yaml(dataset_dir, yaml_path)
    labeled_eval_dir, labeled_eval_yaml, _ = make_labeled_only_eval_dataset(dataset_dir, exp["key"])
    healthy_eval_dir, healthy_eval_yaml, _ = make_healthy_only_eval_dataset(dataset_dir, exp["key"])

    split_counts = {}
    for split in ["train", "valid", "test"]:
        split_counts[split] = count_labeled_images(dataset_dir / split / "labels")
        print(f"{exp['key']} {split}: {split_counts[split]}")

    run_name = f"{RUN_BASE_NAME}_{exp['key']}"
    yolo = make_model(exp)

    start = time.time()
    yolo.train(
        data=str(yaml_path),
        task="segment",
        imgsz=640,
        epochs=100,
        batch=16,
        patience=30,
        seed=42,
        deterministic=True,
        workers=0,
        project=str(RUNS_DIR),
        name=run_name,
        exist_ok=True,
        pretrained=True,
        plots=True,
        verbose=True,
        **BASELINE_TRAIN_ARGS,
    )
    train_time_min = (time.time() - start) / 60

    run_path = RUNS_DIR / run_name
    best_path = run_path / "weights" / "best.pt"
    best_model = YOLO(str(best_path))

    # YOLO val-like reports
    full_val = best_model.val(data=str(yaml_path), split="val", imgsz=640, plots=True, verbose=False)
    full_test = best_model.val(data=str(yaml_path), split="test", imgsz=640, plots=True, verbose=False)
    labeled_val = best_model.val(data=str(labeled_eval_yaml), split="val", imgsz=640, plots=False, verbose=False)
    labeled_test = best_model.val(data=str(labeled_eval_yaml), split="test", imgsz=640, plots=False, verbose=False)

    test_count = count_prediction_errors(best_model, dataset_dir / "test" / "images", dataset_dir / "test" / "labels")
    labeled_test_count = count_prediction_errors(best_model, labeled_eval_dir / "test" / "images", labeled_eval_dir / "test" / "labels")
    healthy_test_fp = healthy_false_positive_summary(best_model, healthy_eval_dir / "test" / "images")

    extra_dir = Path(run_path) / "extra_test_metrics"
    class_metrics = class_level_test_analysis(best_model, dataset_dir, yaml_path, extra_dir)
    save_loss_and_correlation_plots(run_path, extra_dir)

    labeled_mask_map50 = metric_value(labeled_test, "seg.map50")
    count_penalty = COUNT_PENALTY_WEIGHT * labeled_test_count["mask_count_mae"]
    disease_miss_penalty = DISEASE_MISS_PENALTY_WEIGHT * labeled_test_count["disease_box_miss_rate"]
    healthy_fp_penalty = HEALTHY_FP_PENALTY_WEIGHT * healthy_test_fp["healthy_mask_fp_rate"]
    healthy_aware_score = labeled_mask_map50 - count_penalty - disease_miss_penalty - healthy_fp_penalty

    row = {
        "experiment": exp["key"],
        "name": exp["name"],
        "model_type": exp["model_type"],
        "model": exp.get("yaml", "yolo11n-seg.pt"),
        "run_name": run_name,
        "run_path": str(run_path),
        "best_pt": str(best_path),
        "train_time_min": round(train_time_min, 2),

        # Full val/test YOLO report style
        "full_val_box_precision": metric_value(full_val, "box.mp"),
        "full_val_box_recall": metric_value(full_val, "box.mr"),
        "full_val_box_map50": metric_value(full_val, "box.map50"),
        "full_val_box_map50_95": metric_value(full_val, "box.map"),
        "full_val_mask_precision": metric_value(full_val, "seg.mp"),
        "full_val_mask_recall": metric_value(full_val, "seg.mr"),
        "full_val_mask_map50": metric_value(full_val, "seg.map50"),
        "full_val_mask_map50_95": metric_value(full_val, "seg.map"),

        "full_test_box_precision": metric_value(full_test, "box.mp"),
        "full_test_box_recall": metric_value(full_test, "box.mr"),
        "full_test_box_map50": metric_value(full_test, "box.map50"),
        "full_test_box_map50_95": metric_value(full_test, "box.map"),
        "full_test_mask_precision": metric_value(full_test, "seg.mp"),
        "full_test_mask_recall": metric_value(full_test, "seg.mr"),
        "full_test_mask_map50": metric_value(full_test, "seg.map50"),
        "full_test_mask_map50_95": metric_value(full_test, "seg.map"),

        # Labeled-only diseased test
        "labeled_val_box_map50": metric_value(labeled_val, "box.map50"),
        "labeled_val_mask_map50": metric_value(labeled_val, "seg.map50"),
        "labeled_test_box_map50": metric_value(labeled_test, "box.map50"),
        "labeled_test_mask_map50": labeled_mask_map50,
        "labeled_test_mask_map50_95": metric_value(labeled_test, "seg.map"),

        # Count/miss/FP
        "test_gt_instances": test_count["gt_total"],
        "test_pred_boxes": test_count["pred_box_total"],
        "test_pred_masks": test_count["pred_mask_total"],
        "test_mask_count_mae": test_count["mask_count_mae"],
        "labeled_test_gt_instances": labeled_test_count["gt_total"],
        "labeled_test_pred_boxes": labeled_test_count["pred_box_total"],
        "labeled_test_pred_masks": labeled_test_count["pred_mask_total"],
        "labeled_test_mask_count_mae": labeled_test_count["mask_count_mae"],
        "labeled_test_mask_count_exact": labeled_test_count["mask_count_exact"],
        "labeled_test_disease_box_miss_rate": labeled_test_count["disease_box_miss_rate"],
        "labeled_test_disease_mask_miss_rate": labeled_test_count["disease_mask_miss_rate"],
        "labeled_test_disease_box_miss_images": labeled_test_count["disease_box_miss_images"],
        "labeled_test_disease_mask_miss_images": labeled_test_count["disease_mask_miss_images"],
        "healthy_test_images": healthy_test_fp["healthy_images"],
        "healthy_test_mask_fp_rate": healthy_test_fp["healthy_mask_fp_rate"],
        "healthy_test_box_fp_rate": healthy_test_fp["healthy_box_fp_rate"],
        "healthy_test_fp_masks_total": healthy_test_fp["healthy_fp_masks_total"],
        "healthy_test_fp_masks_per_image": healthy_test_fp["healthy_fp_masks_per_image"],
        "healthy_test_avg_fp_confidence": healthy_test_fp["healthy_avg_fp_confidence"],

        # Extra requested metrics
        **class_metrics,

        # Healthy-aware score
        "count_penalty": count_penalty,
        "disease_miss_penalty": disease_miss_penalty,
        "healthy_fp_penalty": healthy_fp_penalty,
        "healthy_aware_labeled_test_mask_map50": healthy_aware_score,
    }
    row.update(read_best_epoch_from_results(run_path))

    del yolo, best_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return row

# =========================================================
# Run all experiments in this notebook
# =========================================================
experiment_results = []
for experiment in EXPERIMENTS:
    result = run_experiment(experiment)
    experiment_results.append(result)
    partial_df = pd.DataFrame(experiment_results)
    display(partial_df)
    partial_df.to_csv(REPORT_DIR / "partial_summary.csv", index=False)

summary_df = pd.DataFrame(experiment_results)
summary_df = summary_df.sort_values("healthy_aware_labeled_test_mask_map50", ascending=False).reset_index(drop=True)
summary_csv = REPORT_DIR / "summary_all_models.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"Saved summary: {summary_csv}")
display(summary_df)

BEST_RUN = summary_df.iloc[0].to_dict()
print("Best run by healthy-aware score:", BEST_RUN["run_name"])
print("Best checkpoint:", BEST_RUN["best_pt"])


## 10. Visualize summary and locate reports

In [ ]:
# =========================================================
# Visualize TEST predictions for EACH module like baseline
# =========================================================
from ultralytics import YOLO
import glob
import matplotlib.pyplot as plt
import os
import random
from pathlib import Path
import cv2
import numpy as np
import pandas as pd

# =========================================================
# Config
# =========================================================
SHOW_ONLY_BEST = False       # True = chỉ show model tốt nhất, False = show từng module
MAX_AUG_IMAGES = 6
MAX_LABELED_TEST_IMAGES = 8
MAX_HEALTHY_TEST_IMAGES = 8
PRED_CONF = 0.10
IMG_SIZE = 640
SEED = 42

random.seed(SEED)

# =========================================================
# Load summary
# =========================================================
summary_csv = REPORT_DIR / "summary_all_models.csv"

if not summary_csv.exists():
    raise FileNotFoundError(
        f"Không tìm thấy summary file: {summary_csv}\n"
        "Hãy chạy cell train/evaluate trước để tạo summary_all_models.csv."
    )

summary_df = pd.read_csv(summary_csv)
display(summary_df)

if SHOW_ONLY_BEST:
    summary_df = summary_df.head(1)

# =========================================================
# Helpers
# =========================================================
def draw_yolo_segmentation_labels(image_path, label_path):
    image = cv2.imread(str(image_path))
    if image is None:
        raise FileNotFoundError(image_path)

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]
    overlay = image.copy()
    colors = [(255, 70, 70), (70, 180, 255), (90, 220, 120), (240, 180, 60)]

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) < 7:
                continue

            cls_id = int(float(parts[0]))
            coords = np.array([float(v) for v in parts[1:]], dtype=np.float32).reshape(-1, 2)
            coords[:, 0] *= width
            coords[:, 1] *= height
            pts = coords.astype(np.int32)

            color = colors[cls_id % len(colors)]
            cv2.polylines(overlay, [pts], isClosed=True, color=color, thickness=2)
            cv2.fillPoly(overlay, [pts], color=color)

            x, y = pts[0]

            try:
                label = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
            except Exception:
                label = str(cls_id)

            cv2.putText(
                overlay,
                label,
                (int(x), int(y)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                color,
                2,
            )

    return cv2.addWeighted(overlay, 0.35, image, 0.65, 0)


def has_nonempty_label(image_path, labels_dir):
    label_path = labels_dir / f"{image_path.stem}.txt"
    return label_path.exists() and bool(label_path.read_text().strip())


def sample_or_first(items, max_n, random_sample=False):
    items = list(items)
    if len(items) <= max_n:
        return items

    if random_sample:
        return random.sample(items, k=max_n)

    return items[:max_n]


# =========================================================
# Loop through each module/model
# =========================================================
for _, row in summary_df.iterrows():
    experiment = row["experiment"]
    run_name = row["run_name"]
    run_path = Path(row["run_path"])
    best_model_path = Path(row["best_pt"])

    selected_exp = experiment
    selected_dataset_dir = EXPERIMENT_ROOT / selected_exp / "dataset"
    selected_yaml = selected_dataset_dir / "data.yaml"

    selected_labeled_yaml = (
        EXPERIMENT_ROOT
        / selected_exp
        / "dataset_labeled_only_eval"
        / "data_labeled_only.yaml"
    )

    selected_healthy_dir = (
        EXPERIMENT_ROOT
        / selected_exp
        / "dataset_healthy_only_eval"
    )

    selected_healthy_yaml = selected_healthy_dir / "data_healthy_only.yaml"

    print("\n" + "=" * 120)
    print(f"Experiment: {experiment}")
    print(f"Run name: {run_name}")
    print(f"Best model: {best_model_path}")
    print(f"Dataset: {selected_dataset_dir}")
    print("=" * 120)

    if not best_model_path.exists():
        print(f"[SKIP] Missing best.pt: {best_model_path}")
        continue

    if not selected_dataset_dir.exists():
        print(f"[SKIP] Missing dataset dir: {selected_dataset_dir}")
        continue

    model_inference = YOLO(str(best_model_path))

    # =====================================================
    # 1. Full test-set evaluation
    # =====================================================
    print("\nFull test-set evaluation:")
    full_test_metrics = model_inference.val(
        data=str(selected_yaml),
        split="test",
        imgsz=IMG_SIZE,
        plots=True,
        verbose=False,
    )

    # =====================================================
    # 2. Labeled-only diseased test-set evaluation
    # =====================================================
    print("\nLabeled-only diseased test-set evaluation:")
    labeled_test_metrics = model_inference.val(
        data=str(selected_labeled_yaml),
        split="test",
        imgsz=IMG_SIZE,
        plots=False,
        verbose=False,
    )

    # =====================================================
    # 3. Healthy false-positive evaluation
    # =====================================================
    healthy_test_fp = healthy_false_positive_summary(
        model_inference,
        selected_healthy_dir / "test" / "images",
    )

    print("Full test mask mAP50:", metric_value(full_test_metrics, "seg.map50"))
    print("Labeled-only diseased test mask mAP50:", metric_value(labeled_test_metrics, "seg.map50"))
    print("Healthy test mask false-positive rate:", healthy_test_fp["healthy_mask_fp_rate"])
    print("Healthy test false-positive masks per image:", healthy_test_fp["healthy_fp_masks_per_image"])

    # =====================================================
    # 4. Visualize augmented training GT masks directly
    # =====================================================
    print("\nVisualize augmented training ground-truth masks directly.")

    aug_images = sorted((selected_dataset_dir / "train" / "images").glob("aug_*"))
    aug_images = sample_or_first(aug_images, MAX_AUG_IMAGES, random_sample=True)

    if aug_images:
        fig, axes = plt.subplots(
            len(aug_images),
            1,
            figsize=(10, 5 * len(aug_images)),
        )

        if len(aug_images) == 1:
            axes = [axes]

        for ax, image_path in zip(axes, aug_images):
            label_path = selected_dataset_dir / "train" / "labels" / f"{image_path.stem}.txt"

            ax.imshow(draw_yolo_segmentation_labels(image_path, label_path))
            ax.set_title(
                f"{experiment} | Augmented train GT mask: {image_path.name} | "
                f"label exists={label_path.exists()}"
            )
            ax.axis("off")

        plt.tight_layout()
        plt.show()
    else:
        print("No augmented training images found to visualize.")

    # =====================================================
    # 5. Visualize labeled test predictions
    # =====================================================
    print("\nVisualize labeled test predictions.")

    test_images = []
    for ext in IMAGE_EXTENSIONS:
        test_images.extend((selected_dataset_dir / "test" / "images").glob(f"*{ext}"))

    test_labels_dir = selected_dataset_dir / "test" / "labels"

    labeled_test_images = [
        p for p in sorted(test_images)
        if has_nonempty_label(p, test_labels_dir)
    ]

    labeled_test_images = sample_or_first(
        labeled_test_images,
        MAX_LABELED_TEST_IMAGES,
        random_sample=False,
    )

    if labeled_test_images:
        results = model_inference.predict(
            source=[str(p) for p in labeled_test_images],
            conf=PRED_CONF,
            imgsz=IMG_SIZE,
            save=True,
            verbose=False,
        )

        for image_path, result in zip(labeled_test_images, results):
            plt.figure(figsize=(8, 8))
            plt.imshow(result.plot())
            plt.title(
                f"{experiment} | Labeled test prediction: {image_path.name} "
                f"(conf > {PRED_CONF})"
            )
            plt.axis("off")
            plt.show()
    else:
        print("No labeled test images found to visualize.")

    # =====================================================
    # 6. Visualize healthy negative predictions
    # =====================================================
    print("\nVisualize healthy negative predictions to inspect false positives.")

    healthy_images = []
    for ext in IMAGE_EXTENSIONS:
        healthy_images.extend((selected_healthy_dir / "test" / "images").glob(f"*{ext}"))

    healthy_images = sample_or_first(
        sorted(healthy_images),
        MAX_HEALTHY_TEST_IMAGES,
        random_sample=False,
    )

    if healthy_images:
        healthy_results = model_inference.predict(
            source=[str(p) for p in healthy_images],
            conf=PRED_CONF,
            imgsz=IMG_SIZE,
            save=False,
            verbose=False,
        )

        for image_path, result in zip(healthy_images, healthy_results):
            box_count = len(result.boxes) if result.boxes is not None else 0
            mask_count = len(result.masks) if result.masks is not None else 0

            plt.figure(figsize=(8, 8))
            plt.imshow(result.plot())
            plt.title(
                f"{experiment} | Healthy test prediction: {image_path.name} | "
                f"boxes={box_count}, masks={mask_count}"
            )
            plt.axis("off")
            plt.show()
    else:
        print("No healthy test images found to visualize.")

print("\nDone visualizing predictions for each module.")

In [ ]:
# =========================================================
# Package all experiment results into a ZIP file
# Works for both single_attention and combined_attention notebooks
# =========================================================
from pathlib import Path
import zipfile
import time
import shutil
import pandas as pd
import os

# =========================================================
# Config
# =========================================================
INCLUDE_WEIGHTS = True      # True = include best.pt/last.pt, False = skip weights to reduce zip size
INCLUDE_DATASET = False     # True = include copied experiment datasets, False = skip datasets
INCLUDE_RUNS = True         # Include YOLO run folders: results.csv, plots, confusion matrix, predictions...
INCLUDE_REPORTS = True      # Include reports folder: summary_all_models.csv, partial_summary.csv...
INCLUDE_EXTRA_METRICS = True

TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")

# =========================================================
# Resolve paths
# =========================================================
if "REPORT_DIR" not in globals():
    raise NameError("REPORT_DIR is not defined. Please run the train/evaluate cell first.")

if "EXPERIMENT_ROOT" not in globals():
    raise NameError("EXPERIMENT_ROOT is not defined. Please run the train/evaluate cell first.")

REPORT_DIR = Path(REPORT_DIR)
EXPERIMENT_ROOT = Path(EXPERIMENT_ROOT)

summary_csv = REPORT_DIR / "summary_all_models.csv"

if not summary_csv.exists():
    raise FileNotFoundError(
        f"summary_all_models.csv not found at: {summary_csv}\n"
        "Please run the train/evaluate cell first."
    )

summary_df = pd.read_csv(summary_csv)

# Detect notebook type from EXPERIMENT_ROOT name
root_name = EXPERIMENT_ROOT.name
zip_name = f"{root_name}_all_results_{TIMESTAMP}.zip"
zip_path = Path("/content") / zip_name

print("Packaging results...")
print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)
print("REPORT_DIR:", REPORT_DIR)
print("summary_csv:", summary_csv)
print("zip_path:", zip_path)

# =========================================================
# Helper functions
# =========================================================
def should_skip_file(path: Path):
    path = Path(path)

    # Skip cache/temp files
    skip_suffixes = {
        ".cache",
        ".tmp",
        ".log",
    }

    if path.suffix.lower() in skip_suffixes:
        return True

    # Skip large weights if disabled
    if not INCLUDE_WEIGHTS and path.suffix.lower() == ".pt":
        return True

    # Skip dataset images/labels if disabled
    if not INCLUDE_DATASET:
        parts = set(path.parts)
        if "dataset" in parts:
            return True
        if "dataset_labeled_only_eval" in parts:
            return True
        if "dataset_healthy_only_eval" in parts:
            return True

    return False


def add_file_to_zip(zf, file_path: Path, arc_base: Path):
    file_path = Path(file_path)

    if not file_path.exists() or not file_path.is_file():
        return

    if should_skip_file(file_path):
        return

    try:
        arcname = file_path.relative_to(arc_base)
    except Exception:
        arcname = file_path.name

    zf.write(file_path, arcname=str(arcname))


def add_dir_to_zip(zf, dir_path: Path, arc_base: Path):
    dir_path = Path(dir_path)

    if not dir_path.exists():
        print(f"[SKIP] Missing dir: {dir_path}")
        return

    for file_path in dir_path.rglob("*"):
        if file_path.is_file():
            add_file_to_zip(zf, file_path, arc_base)


# =========================================================
# Create ZIP
# =========================================================
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:

    # -----------------------------------------------------
    # 1. Add reports folder
    # -----------------------------------------------------
    if INCLUDE_REPORTS and REPORT_DIR.exists():
        print("Adding REPORT_DIR...")
        add_dir_to_zip(zf, REPORT_DIR, REPORT_DIR.parent)

    # -----------------------------------------------------
    # 2. Add experiment root
    # Includes copied datasets only if INCLUDE_DATASET=True
    # -----------------------------------------------------
    if EXPERIMENT_ROOT.exists():
        print("Adding EXPERIMENT_ROOT...")
        add_dir_to_zip(zf, EXPERIMENT_ROOT, EXPERIMENT_ROOT.parent)

    # -----------------------------------------------------
    # 3. Add YOLO run folders from summary_all_models.csv
    # -----------------------------------------------------
    if INCLUDE_RUNS:
        print("Adding YOLO run folders from summary...")

        for _, row in summary_df.iterrows():
            experiment = row.get("experiment", "unknown")
            run_path = Path(row.get("run_path", ""))

            if not run_path.exists():
                print(f"[SKIP] Missing run_path for {experiment}: {run_path}")
                continue

            print(f"  - Adding run: {experiment} -> {run_path}")
            add_dir_to_zip(zf, run_path, run_path.parent)

    # -----------------------------------------------------
    # 4. Add selected important files explicitly
    # -----------------------------------------------------
    important_files = [
        summary_csv,
        REPORT_DIR / "partial_summary.csv",
        REPORT_DIR / "summary_full_test_mask_map50.png",
        REPORT_DIR / "summary_test_missed_box_total.png",
        REPORT_DIR / "summary_test_confused_total.png",
    ]

    for f in important_files:
        if f.exists():
            add_file_to_zip(zf, f, REPORT_DIR.parent)

    # -----------------------------------------------------
    # 5. Add README
    # -----------------------------------------------------
    readme_text = f"""
YOLO11n Attention Experiment Results Package
Generated at: {TIMESTAMP}

Experiment root:
{EXPERIMENT_ROOT}

Report directory:
{REPORT_DIR}

Summary file:
{summary_csv}

Included settings:
- INCLUDE_WEIGHTS = {INCLUDE_WEIGHTS}
- INCLUDE_DATASET = {INCLUDE_DATASET}
- INCLUDE_RUNS = {INCLUDE_RUNS}
- INCLUDE_REPORTS = {INCLUDE_REPORTS}
- INCLUDE_EXTRA_METRICS = {INCLUDE_EXTRA_METRICS}

Main expected files:
- reports/summary_all_models.csv
- reports/partial_summary.csv
- per-run results.csv
- per-run confusion_matrix.png
- per-run results.png
- per-run extra_test_metrics/
- per-run train_val_loss_curves.png
- per-run training_metrics_correlation_matrix.png
- per-run test_predictions_visualized/ if visualization cell was run
- weights/best.pt and weights/last.pt if INCLUDE_WEIGHTS=True
"""

    zf.writestr("README_RESULTS_PACKAGE.txt", readme_text)

# =========================================================
# Report ZIP info
# =========================================================
zip_size_mb = zip_path.stat().st_size / (1024 * 1024)

print("\nDone packaging results.")
print(f"ZIP file: {zip_path}")
print(f"ZIP size: {zip_size_mb:.2f} MB")

print("\nYou can download this file from Kaggle output:")
print(zip_path)

# Optional: show summary table again
print("\nSummary preview:")
display(summary_df)

---
## Phần 2 — Noise Ablation Testing (11 điều kiện)

Đánh giá baseline model trên 11 điều kiện nhiễu thực tế ao nuôi tôm.
Chỉ test images bị thêm noise; train/val vẫn clean để YOLO val() chạy được.


In [ ]:
# =========================================================
# Noise functions — 11 điều kiện thực tế ao nuôi tôm
# N01–N10: từ ablation study v1
# N11: turbidity mới (nước ao đục — khác N05 low contrast)
# =========================================================
import cv2
import numpy as np
from pathlib import Path

SEED = 42
_rng = np.random.default_rng(SEED)
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def apply_gaussian_noise(img, sigma=20):
    """N01: Sensor Gaussian noise — camera rẻ, điều kiện thiếu sáng."""
    noise = _rng.normal(0, sigma, img.shape).astype(np.float32)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)


def apply_salt_pepper(img, prob=0.03):
    """N02: Salt & pepper — dead pixel / sensor fault."""
    out = img.copy()
    mask = _rng.random(img.shape[:2])
    out[mask < prob / 2] = 0
    out[mask > 1 - prob / 2] = 255
    return out


def apply_jpeg_artifact(img, quality=15):
    """N03: JPEG compression artifact q=15 — ảnh gửi qua Zalo/Messenger bị nén mạnh.
    Tạo blocking 8x8 pixel, mất texture chi tiết trên vỏ tôm."""
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, encoded = cv2.imencode('.jpg', img, encode_param)
    return cv2.imdecode(encoded, cv2.IMREAD_COLOR)


def apply_color_cast(img, channel="blue", shift=40):
    """N04: Blue color cast +40 — màu nước ao nuôi tôm thẻ."""
    out = img.astype(np.int32)
    ch_idx = 0 if channel == "blue" else 1  # BGR: B=0, G=1
    out[:, :, ch_idx] = np.clip(out[:, :, ch_idx] + shift, 0, 255)
    return out.astype(np.uint8)


def apply_low_contrast(img, factor=0.5):
    """N05: Low contrast factor=0.5 — toàn ảnh kém tương phản (nước đục nhẹ / thiếu sáng).
    Không giống N11 turbidity: N05 nén tuyến tính về mean, N11 thêm màu sắc đặc trưng ao."""
    mean = np.mean(img, axis=(0, 1), keepdims=True)
    out = mean + (img.astype(np.float32) - mean) * factor
    return np.clip(out, 0, 255).astype(np.uint8)


def apply_gaussian_blur(img, sigma=2.5):
    """N06: Gaussian blur σ=2.5 — tôm di chuyển ra ngoài vùng lấy nét."""
    ksize = int(2 * round(3 * sigma) + 1)
    if ksize % 2 == 0:
        ksize += 1
    return cv2.GaussianBlur(img, (ksize, ksize), sigma)


def apply_heavy_erasing(img, erase_prob=0.5, max_patches=3):
    """N07: Heavy random erasing — che khuất mật độ cao (tôm chồng nhau trong bể)."""
    out = img.copy()
    h, w = img.shape[:2]
    for _ in range(max_patches):
        if _rng.random() < erase_prob:
            ph = int(_rng.uniform(0.1, 0.4) * h)
            pw = int(_rng.uniform(0.1, 0.4) * w)
            y  = int(_rng.uniform(0, h - ph))
            x  = int(_rng.uniform(0, w - pw))
            out[y:y + ph, x:x + pw] = _rng.integers(100, 150, (ph, pw, 3), dtype=np.uint8)
    return out


def apply_motion_blur(img, kernel_size=11):
    """N08: Motion blur kernel=11 — tôm di chuyển nhanh / camera rung."""
    angle = _rng.uniform(0, 180)
    k = np.zeros((kernel_size, kernel_size), dtype=np.float32)
    k[kernel_size // 2, :] = 1.0 / kernel_size
    M = cv2.getRotationMatrix2D((kernel_size // 2, kernel_size // 2), angle, 1)
    k = cv2.warpAffine(k, M, (kernel_size, kernel_size))
    k /= k.sum() if k.sum() > 0 else 1
    return cv2.filter2D(img, -1, k)


def apply_overexposure(img, hsv_v_shift=0.4):
    """N09: Overexposure +40% HSV-V — đèn UV ao nuôi / flash camera chói.
    Khác N10 specular glare: N09 tăng V toàn ảnh đều, N10 tạo điểm sáng cục bộ."""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * (1.0 + hsv_v_shift), 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)


def apply_specular_glare(img, num_patches=3, alpha=0.7):
    """N10: Specular glare — phản chiếu mặt nước / vỏ tôm ướt.
    Nguy hiểm đặc biệt với White Spot Disease: điểm sáng có thể bị nhầm là bệnh đốm trắng."""
    out = img.copy().astype(np.float32)
    h, w = img.shape[:2]
    overlay = out.copy()
    for _ in range(num_patches):
        cx = int(_rng.uniform(0.1, 0.9) * w)
        cy = int(_rng.uniform(0.1, 0.9) * h)
        rx = int(_rng.uniform(0.05, 0.20) * w)
        ry = int(_rng.uniform(0.03, 0.15) * h)
        cv2.ellipse(overlay, (cx, cy), (rx, ry), 0, 0, 360, (255, 255, 255), -1)
    out = cv2.addWeighted(overlay, alpha, out, 1 - alpha, 0)
    return np.clip(out, 0, 255).astype(np.uint8)


def apply_turbidity(img, strength=0.4):
    """N11: Turbidity (nước ao đục) — ao sau mưa / tảo nở rộ.
    Kết hợp haze màu nâu-xám (đặc trưng bùn/tảo) + Gaussian blur nhẹ.
    Khác N05 low_contrast: N11 thêm màu sắc đặc trưng; khác N06 gaussian_blur: N11 thêm haze overlay."""
    # Màu nước đục ao nuôi: nâu xanh (BGR: 120, 140, 160)
    haze = np.full_like(img, [120, 140, 160], dtype=np.float32)
    blurred = cv2.GaussianBlur(img.astype(np.float32), (15, 15), 0)
    out = strength * haze + (1.0 - strength) * blurred
    return np.clip(out, 0, 255).astype(np.uint8)


NOISE_CONDITIONS = [
    ("N01_gaussian",     lambda img: apply_gaussian_noise(img, sigma=20)),
    ("N02_salt_pepper",  lambda img: apply_salt_pepper(img, prob=0.03)),
    ("N03_jpeg_artifact",lambda img: apply_jpeg_artifact(img, quality=15)),
    ("N04_color_cast",   lambda img: apply_color_cast(img, channel="blue", shift=40)),
    ("N05_low_contrast", lambda img: apply_low_contrast(img, factor=0.5)),
    ("N06_gauss_blur",   lambda img: apply_gaussian_blur(img, sigma=2.5)),
    ("N07_heavy_erase",  lambda img: apply_heavy_erasing(img, erase_prob=0.5, max_patches=3)),
    ("N08_motion_blur",  lambda img: apply_motion_blur(img, kernel_size=11)),
    ("N09_overexposure", lambda img: apply_overexposure(img, hsv_v_shift=0.4)),
    ("N10_glare",        lambda img: apply_specular_glare(img, num_patches=3, alpha=0.7)),
    ("N11_turbidity",    lambda img: apply_turbidity(img, strength=0.4)),
]

print(f"Defined {len(NOISE_CONDITIONS)} noise conditions:")
for name, _ in NOISE_CONDITIONS:
    print(" -", name)


In [ ]:
# =========================================================
# Build noisy test sets (11 conditions)
# =========================================================
import shutil
import yaml

NOISE_ROOT = Path("/kaggle/working/noise_test_sets_v3")

# Dùng dataset của experiment baseline (đã copy_dataset_for_experiment)
_base_dataset        = EXPERIMENT_ROOT / "baseline" / "dataset"
_original_test_images = _base_dataset / "test" / "images"
_original_test_labels = _base_dataset / "test" / "labels"

with open(data_yaml_path, "r") as f:
    _base_yaml_content = yaml.safe_load(f)


def build_noisy_test_set(noise_name, noise_fn):
    dst_root         = NOISE_ROOT / noise_name
    dst_test_images  = dst_root / "test" / "images"
    dst_test_labels  = dst_root / "test" / "labels"
    dst_test_images.mkdir(parents=True, exist_ok=True)
    dst_test_labels.mkdir(parents=True, exist_ok=True)

    image_paths = sorted(
        p for p in _original_test_images.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

    for img_path in image_paths:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        cv2.imwrite(str(dst_test_images / img_path.name), noise_fn(img))
        lbl_src = _original_test_labels / f"{img_path.stem}.txt"
        lbl_dst = dst_test_labels / f"{img_path.stem}.txt"
        if lbl_src.exists():
            shutil.copy2(lbl_src, lbl_dst)
        else:
            lbl_dst.write_text("")

    # train/valid: copy clean để YOLO val() không lỗi missing split
    for split in ["train", "valid"]:
        for sub in ["images", "labels"]:
            src_dir = _base_dataset / split / sub
            dst_dir = dst_root / split / sub
            if not dst_dir.exists():
                shutil.copytree(src_dir, dst_dir)

    noise_yaml_content = dict(_base_yaml_content)
    noise_yaml_content["train"] = str(dst_root / "train" / "images")
    noise_yaml_content["val"]   = str(dst_root / "valid" / "images")
    noise_yaml_content["test"]  = str(dst_test_images)
    noise_yaml_path = dst_root / "data.yaml"
    with open(noise_yaml_path, "w") as f:
        yaml.safe_dump(noise_yaml_content, f, sort_keys=False)

    print(f"  Built: {noise_name} ({len(image_paths)} test images)")
    return noise_yaml_path


NOISE_YAML_PATHS = {}
print("Building noisy test sets...")
for noise_name, noise_fn in NOISE_CONDITIONS:
    NOISE_YAML_PATHS[noise_name] = build_noisy_test_set(noise_name, noise_fn)
print(f"Done. {len(NOISE_YAML_PATHS)} noise sets under {NOISE_ROOT}")


In [ ]:
# =========================================================
# Visualize noise samples (sanity check — 11 conditions)
# =========================================================
import matplotlib.pyplot as plt

_sample_paths = sorted(
    p for p in _original_test_images.iterdir()
    if p.suffix.lower() in IMAGE_EXTENSIONS
)[:1]

if _sample_paths:
    _orig_bgr = cv2.imread(str(_sample_paths[0]))
    _orig_rgb = cv2.cvtColor(_orig_bgr, cv2.COLOR_BGR2RGB)

    n_noise = len(NOISE_CONDITIONS)
    n_cols = 4
    n_rows = (n_noise + 1 + n_cols - 1) // n_cols  # ceil
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 4))
    axes = axes.flatten()

    axes[0].imshow(_orig_rgb)
    axes[0].set_title("Original", fontsize=9, fontweight="bold")
    axes[0].axis("off")

    for i, (noise_name, noise_fn) in enumerate(NOISE_CONDITIONS):
        _noisy = cv2.cvtColor(noise_fn(cv2.imread(str(_sample_paths[0]))), cv2.COLOR_BGR2RGB)
        axes[i + 1].imshow(_noisy)
        axes[i + 1].set_title(noise_name, fontsize=8)
        axes[i + 1].axis("off")

    for j in range(n_noise + 1, len(axes)):
        axes[j].axis("off")

    plt.suptitle(f"11 Noise Conditions — {_sample_paths[0].name}", fontsize=12)
    plt.tight_layout()
    plt.savefig(NOISE_ROOT / "noise_samples_overview.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No test images found.")


In [ ]:
# =========================================================
# Evaluate baseline model on all 11 noise conditions
# Dùng lại toàn bộ helper functions từ cell train:
#   metric_value, count_prediction_errors, healthy_false_positive_summary
# =========================================================
import gc
import pandas as pd
import torch
from IPython.display import display
from ultralytics import YOLO

NOISE_REPORT_DIR = EXPERIMENT_ROOT / "reports" / "noise"
NOISE_REPORT_DIR.mkdir(parents=True, exist_ok=True)

PREDICT_CONF = 0.25
all_noise_rows = []

for exp in EXPERIMENTS:
    exp_key = exp["key"]
    best_pt = RUNS_DIR / f"{RUN_BASE_NAME}_{exp_key}" / "weights" / "best.pt"
    if not best_pt.exists():
        print(f"[SKIP] Missing best.pt for {exp_key}: {best_pt}")
        continue

    print(f"\n=== Noise eval: {exp_key} ===")
    yolo = YOLO(str(best_pt))

    for noise_name, _ in NOISE_CONDITIONS:
        noise_yaml         = NOISE_YAML_PATHS[noise_name]
        noise_test_images_dir = NOISE_ROOT / noise_name / "test" / "images"
        noise_test_labels_dir = NOISE_ROOT / noise_name / "test" / "labels"

        # labeled-only subset
        labeled_noise_yaml = NOISE_ROOT / noise_name / "data_labeled.yaml"
        if not labeled_noise_yaml.exists():
            import yaml as _yaml
            import shutil
            with open(noise_yaml) as _f:
                _nc = _yaml.safe_load(_f)
            _ltest = NOISE_ROOT / noise_name / "test_labeled" / "images"
            _ltlbl = NOISE_ROOT / noise_name / "test_labeled" / "labels"
            _ltest.mkdir(parents=True, exist_ok=True)
            _ltlbl.mkdir(parents=True, exist_ok=True)
            for _ip in sorted(noise_test_images_dir.iterdir()):
                if _ip.suffix.lower() not in IMAGE_EXTENSIONS:
                    continue
                _lp = noise_test_labels_dir / f"{_ip.stem}.txt"
                if _lp.exists() and _lp.read_text().strip():
                    shutil.copy2(_ip, _ltest / _ip.name)
                    shutil.copy2(_lp, _ltlbl / _lp.name)
            _nc["test"] = str(_ltest)
            with open(labeled_noise_yaml, "w") as _f:
                _yaml.safe_dump(_nc, _f, sort_keys=False)

        print(f"  {noise_name}...", end=" ", flush=True)
        try:
            full_m = yolo.val(data=str(noise_yaml), split="test",
                              imgsz=640, plots=False, verbose=False)
            full_mask_map50    = metric_value(full_m, "seg.map50")
            full_mask_map50_95 = metric_value(full_m, "seg.map")

            labeled_m = yolo.val(data=str(labeled_noise_yaml), split="test",
                                 imgsz=640, plots=False, verbose=False)
            labeled_mask_map50 = metric_value(labeled_m, "seg.map50")

            # count errors on labeled subset
            _ltest_imgs = NOISE_ROOT / noise_name / "test_labeled" / "images"
            _ltest_lbls = NOISE_ROOT / noise_name / "test_labeled" / "labels"
            cnt = count_prediction_errors(yolo, _ltest_imgs, _ltest_lbls)

            # healthy FP on noisy healthy images
            import shutil as _sh
            _healthy_imgs = []
            for _ip in sorted(noise_test_images_dir.iterdir()):
                if _ip.suffix.lower() not in IMAGE_EXTENSIONS:
                    continue
                _lp = noise_test_labels_dir / f"{_ip.stem}.txt"
                if not (_lp.exists() and _lp.read_text().strip()):
                    _healthy_imgs.append(_ip)
            if _healthy_imgs:
                _hdir = NOISE_ROOT / noise_name / "test_healthy" / "images"
                _hdir.mkdir(parents=True, exist_ok=True)
                for _ip in _healthy_imgs:
                    _dst = _hdir / _ip.name
                    if not _dst.exists():
                        _sh.copy2(_ip, _dst)
                fp = healthy_false_positive_summary(yolo, _hdir)
            else:
                fp = {"healthy_images": 0, "healthy_mask_fp_rate": float("nan"),
                      "healthy_fp_masks_per_image": float("nan"),
                      "healthy_avg_fp_confidence": float("nan")}

            count_penalty   = COUNT_PENALTY_WEIGHT * cnt["mask_count_mae"]
            miss_penalty    = DISEASE_MISS_PENALTY_WEIGHT * cnt["disease_box_miss_rate"]
            healthy_penalty = HEALTHY_FP_PENALTY_WEIGHT * fp["healthy_mask_fp_rate"]
            healthy_aware   = labeled_mask_map50 - count_penalty - miss_penalty - healthy_penalty

        except Exception as e:
            print(f"ERROR: {e}")
            full_mask_map50 = labeled_mask_map50 = full_mask_map50_95 = float("nan")
            count_penalty = miss_penalty = healthy_penalty = healthy_aware = float("nan")
            cnt = {"mask_count_mae": float("nan"), "disease_box_miss_rate": float("nan"),
               "disease_mask_miss_rate": float("nan"), "mask_count_exact": float("nan")}
            fp  = {"healthy_mask_fp_rate": float("nan"), "healthy_fp_masks_per_image": float("nan"),
                   "healthy_avg_fp_confidence": float("nan")}

        all_noise_rows.append({
            "experiment":                  exp_key,
            "model_name":                  exp["name"],
            "noise":                       noise_name,
            "full_test_mask_map50":        full_mask_map50,
            "full_test_mask_map50_95":     full_mask_map50_95,
            "labeled_test_mask_map50":     labeled_mask_map50,
            "labeled_test_mask_count_mae": cnt["mask_count_mae"],
            "labeled_test_mask_count_exact": cnt.get("mask_count_exact", float("nan")),
            "labeled_test_disease_box_miss_rate": cnt["disease_box_miss_rate"],
            "labeled_test_disease_mask_miss_rate": cnt["disease_mask_miss_rate"],
            "healthy_test_mask_fp_rate":   fp["healthy_mask_fp_rate"],
            "healthy_fp_masks_per_image":  fp["healthy_fp_masks_per_image"],
            "healthy_avg_fp_confidence":   fp["healthy_avg_fp_confidence"],
            "count_penalty":               count_penalty,
            "disease_miss_penalty":        miss_penalty,
            "healthy_fp_penalty":          healthy_penalty,
            "healthy_aware_labeled_test_mask_map50": healthy_aware,
        })
        print(f"labeled_map50={labeled_mask_map50:.4f} | healthy_aware={healthy_aware:.4f}")

    del yolo
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

noise_df = pd.DataFrame(all_noise_rows)
noise_df.to_csv(NOISE_REPORT_DIR / "noise_eval_all.csv", index=False)
print("\nAll noise evaluation results:")
display(noise_df)


In [ ]:
# =========================================================
# Summary table: Baseline noise evaluation — 11 conditions
# =========================================================
import pandas as pd
from IPython.display import display

noise_df = pd.read_csv(NOISE_REPORT_DIR / "noise_eval_all.csv")

summary = noise_df[[
    "noise",
    "labeled_test_mask_map50",
    "healthy_aware_labeled_test_mask_map50",
    "labeled_test_disease_box_miss_rate",
    "healthy_test_mask_fp_rate",
    "labeled_test_mask_count_mae",
    "healthy_fp_masks_per_image",
]].round(4)

summary.to_csv(NOISE_REPORT_DIR / "noise_eval_baseline_summary.csv", index=False)
print("Baseline noise evaluation — 11 conditions:")
display(summary)

# Quick stats
print(f"\nMean labeled_test_mask_map50:  {summary['labeled_test_mask_map50'].mean():.4f}")
print(f"Mean healthy_aware score:      {summary['healthy_aware_labeled_test_mask_map50'].mean():.4f}")
print(f"Mean disease miss rate:        {summary['labeled_test_disease_box_miss_rate'].mean():.4f}")
print(f"Mean healthy FP rate:          {summary['healthy_test_mask_fp_rate'].mean():.4f}")


In [ ]:
# =========================================================
# Charts — Baseline on 11 noise conditions
# =========================================================
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

noise_df  = pd.read_csv(NOISE_REPORT_DIR / "noise_eval_all.csv")
clean_df  = pd.read_csv(REPORT_DIR / "summary_all_models.csv")

noise_names = [n for n, _ in NOISE_CONDITIONS]
base_rows = noise_df.set_index("noise")

x = np.arange(len(noise_names))

# ── Fig 1: labeled_test_mask_map50 per noise ──
fig, ax = plt.subplots(figsize=(14, 5))
vals = [base_rows.loc[n, "labeled_test_mask_map50"] if n in base_rows.index else 0
        for n in noise_names]
bars = ax.bar(x, vals, color="steelblue", alpha=0.85)

if "baseline" in clean_df["experiment"].values:
    cv = float(clean_df.loc[clean_df["experiment"]=="baseline", "labeled_test_mask_map50"].iloc[0])
    ax.axhline(cv, color="darkorange", linestyle="--", alpha=0.7,
               label=f"Clean baseline ({cv:.3f})")

for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005,
            f"{v:.3f}", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(noise_names, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("labeled_test_mask_mAP50")
ax.set_title("Baseline YOLO11n-seg — 11 Noise Conditions\n(dashed = clean test)")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(NOISE_REPORT_DIR / "noise_labeled_map50_baseline.png", dpi=200, bbox_inches="tight")
plt.show()

# ── Fig 2: Drop from clean baseline ──
if "baseline" in clean_df["experiment"].values:
    clean_val = float(clean_df.loc[clean_df["experiment"]=="baseline", "labeled_test_mask_map50"].iloc[0])
    drops = [v - clean_val for v in vals]
    colors = ["green" if d >= 0 else "crimson" for d in drops]
    fig2, ax2 = plt.subplots(figsize=(14, 4))
    bars2 = ax2.bar(noise_names, drops, color=colors, alpha=0.85)
    ax2.axhline(0, color="black", linewidth=0.8)
    for bar, d in zip(bars2, drops):
        ax2.text(bar.get_x() + bar.get_width()/2, d + (0.003 if d >= 0 else -0.006),
                 f"{d:+.3f}", ha="center", va="bottom" if d >= 0 else "top", fontsize=8)
    ax2.set_xticklabels(noise_names, rotation=30, ha="right", fontsize=9)
    ax2.set_ylabel("Drop from clean baseline (mAP50)")
    ax2.set_title("Performance Drop per Noise Condition (green=no drop)")
    ax2.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(NOISE_REPORT_DIR / "noise_drop_from_clean.png", dpi=200, bbox_inches="tight")
    plt.show()

# ── Fig 3: healthy-aware score ──
fig3, ax3 = plt.subplots(figsize=(14, 5))
ha_vals = [base_rows.loc[n, "healthy_aware_labeled_test_mask_map50"]
           if n in base_rows.index else 0 for n in noise_names]
ax3.bar(x, ha_vals, color="steelblue", alpha=0.85)
ax3.set_xticks(x)
ax3.set_xticklabels(noise_names, rotation=30, ha="right", fontsize=9)
ax3.set_ylabel("healthy_aware_score")
ax3.set_title("Baseline Healthy-Aware Score — 11 Noise Conditions")
ax3.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(NOISE_REPORT_DIR / "noise_healthy_aware_baseline.png", dpi=200, bbox_inches="tight")
plt.show()

# ── Fig 4: miss rate and FP rate ──
fig4, (ax4a, ax4b) = plt.subplots(1, 2, figsize=(16, 5))
for ax4, metric, title in [
    (ax4a, "labeled_test_disease_mask_miss_rate", "Disease Miss Rate (lower=better)"),
    (ax4b, "healthy_test_mask_fp_rate",           "Healthy FP Rate (lower=better)"),
]:
    metric_vals = [base_rows.loc[n, metric] if n in base_rows.index else 0
                   for n in noise_names]
    ax4.bar(x, metric_vals, color="steelblue", alpha=0.85)
    ax4.set_xticks(x)
    ax4.set_xticklabels(noise_names, rotation=30, ha="right", fontsize=8)
    ax4.set_title(title)
    ax4.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(NOISE_REPORT_DIR / "noise_miss_fp_rates_baseline.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# =========================================================
# Visualize predictions: Baseline on noisy test images
# =========================================================
import matplotlib.pyplot as plt
import cv2
import gc
import torch
from ultralytics import YOLO

MAX_IMAGES_PER_NOISE = 2   # 2 images × 11 noises × 3 cols = manageable
PRED_CONF_VIZ = 0.15

model_paths = {}
for exp in EXPERIMENTS:
    bp = RUNS_DIR / f"{RUN_BASE_NAME}_{exp['key']}" / "weights" / "best.pt"
    if bp.exists():
        model_paths[exp["key"]] = (str(bp), exp["name"])

for noise_name, _ in NOISE_CONDITIONS:
    noise_test_images_dir = NOISE_ROOT / noise_name / "test" / "images"
    all_imgs = sorted(
        p for p in noise_test_images_dir.iterdir()
        if p.suffix.lower() in IMAGE_EXTENSIONS
    )[:MAX_IMAGES_PER_NOISE]
    if not all_imgs:
        continue

    n_cols = len(model_paths) + 1
    fig, axes = plt.subplots(len(all_imgs), n_cols,
                             figsize=(6 * n_cols, 5 * len(all_imgs)))
    if len(all_imgs) == 1:
        axes = [axes]

    for row_idx, img_path in enumerate(all_imgs):
        orig_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        axes[row_idx][0].imshow(orig_rgb)
        axes[row_idx][0].set_title(f"Noisy: {img_path.name[:26]}", fontsize=8)
        axes[row_idx][0].axis("off")

        for col_idx, (exp_key, (model_pt, model_name)) in enumerate(model_paths.items(), start=1):
            _yolo = YOLO(model_pt)
            result = _yolo.predict(str(img_path), imgsz=640,
                                   conf=PRED_CONF_VIZ, verbose=False)[0]
            mask_count = len(result.masks) if result.masks is not None else 0
            axes[row_idx][col_idx].imshow(result.plot())
            axes[row_idx][col_idx].set_title(
                f"{exp_key} | masks={mask_count}", fontsize=8)
            axes[row_idx][col_idx].axis("off")
            del _yolo
            gc.collect()

    plt.suptitle(f"Noise: {noise_name}", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(NOISE_REPORT_DIR / f"viz_{noise_name}.png", dpi=130, bbox_inches="tight")
    plt.show()

print("Visualization done.")


In [ ]:
# =========================================================
# Package: ZIP with clean + noise results
# =========================================================
import zipfile, time
from pathlib import Path
import pandas as pd

TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
root_name = EXPERIMENT_ROOT.name
zip_path = Path("/content") / f"{root_name}_all_results_{TIMESTAMP}.zip"

INCLUDE_WEIGHTS = True
INCLUDE_DATASET = False

def _skip(p):
    p = Path(p)
    if p.suffix.lower() in {".cache", ".tmp", ".log"}:
        return True
    if not INCLUDE_WEIGHTS and p.suffix.lower() == ".pt":
        return True
    if not INCLUDE_DATASET and any(d in p.parts for d in
            ["dataset", "dataset_labeled_only_eval", "dataset_healthy_only_eval"]):
        return True
    return False

def _add_dir(zf, dir_path, arc_base):
    for fp in Path(dir_path).rglob("*"):
        if fp.is_file() and not _skip(fp):
            try:
                zf.write(fp, arcname=str(fp.relative_to(arc_base)))
            except Exception:
                zf.write(fp, arcname=fp.name)

summary_csv = REPORT_DIR / "summary_all_models.csv"
summary_df  = pd.read_csv(summary_csv)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    if REPORT_DIR.exists():
        _add_dir(zf, REPORT_DIR, REPORT_DIR.parent)
    if EXPERIMENT_ROOT.exists():
        _add_dir(zf, EXPERIMENT_ROOT, EXPERIMENT_ROOT.parent)
    for _, row in summary_df.iterrows():
        rp = Path(row.get("run_path", ""))
        if rp.exists():
            _add_dir(zf, rp, rp.parent)
    if NOISE_REPORT_DIR.exists():
        _add_dir(zf, NOISE_REPORT_DIR, NOISE_REPORT_DIR.parent)
    if (NOISE_ROOT / "noise_samples_overview.png").exists():
        zf.write(NOISE_ROOT / "noise_samples_overview.png",
                 arcname="reports/noise/noise_samples_overview.png")

    readme = f"""YOLO11n Baseline — Noise Ablation Study Results
Generated: {TIMESTAMP}

Pipeline: baseline-only noise evaluation
Noise conditions (11): {[n for n, _ in NOISE_CONDITIONS]}

Key files:
  reports/summary_all_models.csv                         — clean eval (Baseline)
  reports/noise/noise_eval_all.csv                       — all noise evaluations (raw)
  reports/noise/noise_eval_baseline_summary.csv          — baseline summary per noise
  reports/noise/noise_samples_overview.png               — visual overview 11 noise types
  reports/noise/noise_labeled_map50_baseline.png         — bar per noise
  reports/noise/noise_drop_from_clean.png                — drop from clean baseline
  reports/noise/noise_healthy_aware_baseline.png         — healthy-aware score
  reports/noise/noise_miss_fp_rates_baseline.png         — miss rate & FP rate
  reports/noise/viz_<noise>.png                          — prediction visualizations
"""
    zf.writestr("README_NOISE_ABLATION_BASELINE.txt", readme)

zip_size_mb = zip_path.stat().st_size / (1024 * 1024)
print(f"ZIP: {zip_path}")
print(f"Size: {zip_size_mb:.2f} MB")

print("\n" + "=" * 70)
print("CLEAN EVAL SUMMARY")
print("=" * 70)
display(pd.read_csv(REPORT_DIR / "summary_all_models.csv")[
    ["experiment", "labeled_test_mask_map50", "healthy_aware_labeled_test_mask_map50",
     "labeled_test_disease_box_miss_rate", "healthy_test_mask_fp_rate", "train_time_min"]
])

print("\n" + "=" * 70)
print("NOISE EVAL SUMMARY (11 conditions)")
print("=" * 70)
noise_summary = pd.read_csv(NOISE_REPORT_DIR / "noise_eval_baseline_summary.csv")
display(noise_summary)
print(f"\nMean labeled_test_mask_map50: {noise_summary['labeled_test_mask_map50'].mean():.4f}")
print(f"Mean healthy_aware score:     {noise_summary['healthy_aware_labeled_test_mask_map50'].mean():.4f}")
